# v2.10 — v2.06 with class-weighted cross-entropy (3-class k×σ label, horizons 1 / 5 / 10; validation only)

**One change on top of v2.06** (Siwat, 2026-11-08, `configs/v2_10.json`):
- **What changes:** the trend cross-entropy weights each row by its class, one weight set per horizon: **(1 / (3 × the class's training share))²**, class order down / flat / up:
  - 1 d 0.9299 / 0.9960 / 1.0829
  - 5 d 0.9372 / 0.9992 / 1.0702
  - 10 d 0.9048 / 1.0019 / 1.1089

  The labels are near-balanced (about 34.6 / 33.4 / 32.0%), so the weights push gently toward "up" and away from "down". The script recomputes the shares and stops if they differ.
- **What stays:** everything else is v2.06: no price head, news-robust training (p 0.2), max 30 epochs, the same rows and seeds.

**Ablation:** like v2.06, no price head (proposal L5 / L6).

**Runtime:** A100 or L4, Colab only (Linux). Runtime → Run all. About 1 h of training, then about 15 min for the post-hoc step and the two comparisons.

**Adopt rule** (pre-registered, `claude/log-entry-2026-11-08-v2-10-design.md`; the rule v2.06 failed), raw models vs **v2.04**:
- **(a)** news on: the paired 95% interval of v2.10 − v2.04 (3-horizon mean) is entirely above 0;
- **(b)** news blanked: that interval is not entirely below 0.

v2.10 vs v2.06 is printed after it, reported only.

Outputs: `Senior/results/v2xx/<GPU>/v2_10*`, `Senior/results/v2_10_posthoc/`, `Senior/results/v2_10_vs_v2_04.json`, `Senior/results/v2_10_vs_v2_06.json`. Test block sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
from pathlib import Path
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
OUT = f'{RES}/v2xx/{GPU_TAG}'
V210 = f'{OUT}/v2_10.json'
POST = f'{RES}/v2_10_posthoc'                      # v2.10's post-hoc step
B06 = f'{RES}/v2_06_posthoc'                       # v2.06's post-hoc step (reported comparison)
BASE = f'{RES}/v2_04_posthoc'                      # v2.04's post-hoc step, the comparison
BASE_RUN = f'{RES}/v2xx/A100/v2_04.json'           # v2.04's run (A100, 2026-10-27)
FWD_INPUTS = f'{RES}/forward_2026/inputs'
for f in [f'{B06}/step0.json', f'{BASE}/step0.json', f'{BASE}/trees.csv.gz', f'{BASE}/parts/val_blank.csv.gz', BASE_RUN, f'{FWD_INPUTS}/forward_prices.csv',
          f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv']:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)

def stream(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
for f in ['configs/v2_10.json', 'scripts/train_v2xx.py', 'src/train_utils.py', 'src/run_config.py', 'tests/test_no_price_head.py', 'tests/test_v2_10.py', 'src/proposal_model.py',
          'scripts/step0_v2_03beta.py', 'scripts/compare_v2xx_posthoc.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.10 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the frozen copy'
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
import run_config as RC
cfg = RC.load('configs/v2_10.json')
print('v2.10 class weights:', cfg['loss']['class_weights'], '| price head:', cfg['model']['price_head'], '| loss:', json.dumps(cfg['loss']), '| training:', json.dumps(cfg['training']))

In [ ]:
# Tests (about 12 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_news_blank.py', 'tests/test_grad_diag.py', 'tests/test_no_price_head.py',
          'tests/test_step0_v2_03beta.py', 'tests/test_v2_10.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l and '    seed' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 1. Train v2.10 (v2.06 + per-horizon class weights): 3 seeds one after another
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
if done(V210):
    print('v2.10: already trained')
else:
    stream(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_10.json', '--out', V210])

In [ ]:
# 2. Summary beside v2.04 and the floors (no price metrics: no price head), and the learning curves
r = json.load(open(V210)); s, fl = r['summary']['model'], r['floors']
b = json.load(open(BASE_RUN))['summary']['model']
print('class weights per horizon', r['loss_class_weights'])
BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
print(f"{'':6}{'v2.10 F1':>18}{'v2.04':>8}{'bar':>7}{'continues':>11}{'reverses':>10}")
for h in ('h1', 'h5', 'h10'):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>11.4f} ± {m['macro_f1']['std']:.4f}{b[h]['macro_f1']['mean']:>8.4f}{BAR[h]:>7.3f}"
          f"{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}")
for x in r['runs']:
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'hold-out F1', x['best_inner_es_macro_f1'],
          'weights', x['params']['total'], 'minutes', round(x['seconds'] / 60, 1))
from IPython.display import Image, display
import importlib, plot_v2xx_training as PT
for png in (PT.plot(Path(V210)), PT.plot_calls(Path(V210))):
    if png:
        print('saved', png); display(Image(filename=str(png)))

In [ ]:
# 3. Post-hoc step for v2.10 (same script and settings as v2.03beta-v2.07): reproduction check, ts, blend (w on v2.10's
#    own inner hold-out), news blanked and placebo, on validation and 2026
common = ['--run-json', V210, '--fwd-inputs', FWD_INPUTS, '--out-dir', POST, '--label', 'v2.10 post-hoc']
for stage in ('prepare', 'predict', 'trees', 'analyze'):
    stream(['python', 'scripts/step0_v2_03beta.py', stage, *common])

In [ ]:
# 4. v2.10 vs v2.04 on the same rows, and the pre-registered adopt rule (news-on gain significant; blanked held)
stream(['python', 'scripts/compare_v2xx_posthoc.py', '--base', BASE, '--new', POST, '--rule', 'trend_up_significant',
        '--out', f'{RES}/v2_10_vs_v2_04.json'])

In [ ]:
# 5. REPORTED ONLY (not the adopt rule): v2.10 vs v2.06, the effect of the class weights alone.
#    Ignore the ADOPT RULE line this prints; the rule above (vs v2.04) decides.
stream(['python', 'scripts/compare_v2xx_posthoc.py', '--base', B06, '--new', POST, '--rule', 'trend_up_significant',
        '--out', f'{RES}/v2_10_vs_v2_06.json'])

**Reading the result**
- **ADOPT RULE** in block 4 (vs v2.04): (a) the news-on gain is significant (interval entirely above 0), and (b) news blanked doesn't fall significantly. Raw models.
- Block 5 (vs v2.06) is the class weights' own effect, reported only; its ADOPT line does not count.
- Watch the **call shares**: v2.07 (1 d) over-called "flat" and under-called "up"; the weights favour "up" slightly.
- Send the printout to Claude.